# 04 — V1 vs V2 Comparison Analysis

This notebook compares the centralized (V1) model against the federated (V2) model:
- Performance metrics comparison
- Privacy analysis
- Communication cost
- Trade-off recommendations

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.config import MODELS_DIR, PLOTS_DIR, REPORTS_DIR, N_CLIENTS, NUM_ROUNDS
from src.data_utils import load_client_data, prepare_client_tensors, get_global_scaler, load_dataset
from src.models import get_model_parameters, get_logistic_regression
from src.metrics import evaluate_model, federated_vs_centralized_report, compute_communication_cost

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Models

In [ ]:
# V2: Global federated model
with open(MODELS_DIR / 'global_model.pkl', 'rb') as f:
    federated_model = pickle.load(f)
print('Federated model loaded.')

# V1: Centralized baseline — train on all data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = load_dataset()
feature_cols = [c for c in df.columns if c != 'Diabetes_binary']
X = df[feature_cols].values.astype(np.float32)
y = df['Diabetes_binary'].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_train_c)
X_test_c = scaler_c.transform(X_test_c)

centralized_model = get_logistic_regression({'warm_start': False})
centralized_model.fit(X_train_c, y_train_c)
print('Centralized model trained.')

## 2. Evaluate Both Models

In [ ]:
# Centralized on held-out test
v1_metrics = evaluate_model(centralized_model, X_test_c, y_test_c, label='Centralized (V1)')

# Federated on all clients' test sets combined
client_dfs = {i: load_client_data(i) for i in range(N_CLIENTS)}
scaler_fed = get_global_scaler(client_dfs)

X_all_test, y_all_test = [], []
for cid, cdf in client_dfs.items():
    _, _, X_test, y_test, _ = prepare_client_tensors(cdf, scaler=scaler_fed)
    X_all_test.append(X_test)
    y_all_test.append(y_test)

X_all_test = np.vstack(X_all_test)
y_all_test = np.concatenate(y_all_test)

v2_metrics = evaluate_model(federated_model, X_all_test, y_all_test, label='Federated (V2)')

print()
report = federated_vs_centralized_report(v2_metrics, v1_metrics)
print(report)

## 3. Comparison Bar Chart

In [ ]:
metrics = ['accuracy', 'f1', 'roc_auc']
v1_vals = [v1_metrics[m] for m in metrics]
v2_vals = [v2_metrics[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, v1_vals, width, label='Centralized (V1)', color='steelblue')
b2 = ax.bar(x + width/2, v2_vals, width, label='Federated (V2)', color='darkorange')
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics])
ax.set_ylabel('Score')
ax.set_title('V1 (Centralized) vs V2 (Federated) — Performance Comparison', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'v1_v2_comparison.png', bbox_inches='tight')
plt.show()

## 4. System Properties Comparison

In [ ]:
params = get_model_parameters(federated_model)
comm = compute_communication_cost(N_CLIENTS, NUM_ROUNDS, params)

comparison_table = pd.DataFrame({
    'Property': [
        'Accuracy', 'F1 Score', 'ROC-AUC',
        'Data Privacy', 'Communication Cost',
        'Scalability', 'Data Centralisation Required',
        'Suitable for Healthcare'
    ],
    'Centralized (V1)': [
        f"{v1_metrics['accuracy']:.4f}",
        f"{v1_metrics['f1']:.4f}",
        f"{v1_metrics['roc_auc']:.4f}",
        'None — raw data centralised',
        'N/A',
        'Poor — requires data movement',
        'Yes',
        'Limited (privacy concerns)'
    ],
    'Federated (V2)': [
        f"{v2_metrics['accuracy']:.4f}",
        f"{v2_metrics['f1']:.4f}",
        f"{v2_metrics['roc_auc']:.4f}",
        'High — data stays local',
        f"{comm['total_mb']:.4f} MB",
        'High — scales to any # of clients',
        'No',
        'Yes (HIPAA / GDPR aligned)'
    ]
})
comparison_table = comparison_table.set_index('Property')
print(comparison_table.to_string())

## 5. Convergence Overlay

In [ ]:
with open(MODELS_DIR / 'fl_metrics_history.json') as f:
    fl_data = json.load(f)
fl_history = fl_data['fl_metrics_history']

rounds = [h['round'] for h in fl_history]
accs = [h.get('accuracy', 0.0) for h in fl_history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rounds, accs, marker='o', color='darkorange', label='Federated (V2) per round', linewidth=2)
ax.axhline(v1_metrics['accuracy'], linestyle='--', color='steelblue', linewidth=2,
           label=f'Centralized (V1) = {v1_metrics["accuracy"]:.4f}')
ax.set_xlabel('FL Round')
ax.set_ylabel('Accuracy')
ax.set_title('Convergence: Federated vs. Centralized Baseline', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'convergence_vs_centralized.png', bbox_inches='tight')
plt.show()

## 6. Save Comparison Report

In [ ]:
comparison_data = {
    'centralized': v1_metrics,
    'federated': v2_metrics,
    'communication_cost': comm,
}

with open(REPORTS_DIR / 'v1_v2_comparison.json', 'w') as f:
    json.dump(comparison_data, f, indent=2)

comparison_table.to_csv(REPORTS_DIR / 'v1_v2_comparison_table.csv')
print('Comparison report saved to results/reports/')

## 7. Key Findings & Recommendations

### Why V2 may have lower accuracy:
- **Non-IID data**: Each client sees a different data distribution, making the aggregated model a compromise
- **Limited local training**: Only 1 local epoch per round reduces local fitting
- **Aggregation effects**: FedAvg averages weights which may not perfectly align across heterogeneous clients

### When to prefer Federated (V2):
- Data is distributed across organisations (hospitals, clinics)
- Privacy regulations prevent data sharing (HIPAA, GDPR)
- Data volumes are too large to centralise
- Organisations want to retain data ownership

### When to prefer Centralized (V1):
- Data can be legally and securely centralised
- Maximum model accuracy is the priority
- Small-scale projects with no privacy constraints